In [ ]:
# 1/4 - clone CUDA backend (HEAD c7bf7db: warp-shuffle LN)
import subprocess, os
d='/content/TokenVectorCuda'
subprocess.run('rm -rf '+d, shell=True)
subprocess.run('git clone --depth 1 --branch prototype/cuda-backend https://github.com/nguyenhungtran18/TokenVector.git '+d, shell=True, capture_output=True, text=True, timeout=600)
ex=d+'/Release/cuda_backend/examples'
print('HEAD', subprocess.check_output('git -C '+d+' rev-parse --short HEAD', shell=True).decode().strip())
for n in ['layernorm','layernorm_shared','layernorm_warp','layernorm_warp1']:
    s=open(ex+'/'+n+'.cuda.cu').read()
    print(n,'has wreduce =', 'tkv_wreduce' in s)

In [ ]:
# 2/4 - build all 3 .so
import subprocess, shutil, os
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
nvcc=shutil.which('nvcc') or '/usr/local/cuda/bin/nvcc'
for name in ['layernorm','layernorm_shared','layernorm_warp','layernorm_warp1']:
    cmd=(nvcc+' -O2 -shared -std=c++20 --expt-relaxed-constexpr -x cu -Xcompiler -std=c++20 -Xcompiler -fPIC '+ex+'/'+name+'.cuda.cu -o '+ex+'/'+name+'.so -lcudart')
    p=subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600)
    print(name,'build rc=',p.returncode)
    if p.returncode!=0: print(p.stderr[:1500])

In [ ]:
# 3/4 - correctness + speed: ATen / naive / shared / warp / warp1
import ctypes, torch, time
ex='/content/TokenVectorCuda/Release/cuda_backend/examples'
F32=ctypes.c_float; I32=ctypes.c_int; I64=ctypes.c_int64
def load(name, sym):
    lib=ctypes.CDLL(ex+'/'+name+'.so'); fn=getattr(lib,sym)
    fn.argtypes=[I64,I64,I64,I64,I32,I32,F32,I32,I32]; fn.restype=I32
    return fn
naive=load('layernorm','tkv_launch_layernorm_fwd')
shared=load('layernorm_shared','tkv_launch_layernorm_shared')
warp=load('layernorm_warp','tkv_launch_layernorm_warp')
warp1=load('layernorm_warp1','tkv_launch_layernorm_warp1')
h=256; rows=4096
x=torch.randn(rows,h,device='cuda',dtype=torch.float32)
g=torch.randn(h,device='cuda',dtype=torch.float32); b=torch.randn(h,device='cuda',dtype=torch.float32)
ref=torch.nn.functional.layer_norm(x,(h,),g,b,1e-6)
for nm,fn,blk in [('naive',naive,256),('shared',shared,256),('warp',warp,256),('warp1',warp1,32)]:
    o=torch.empty_like(x)
    rc=fn(x.data_ptr(),g.data_ptr(),b.data_ptr(),o.data_ptr(),rows,h,1e-6,rows,blk)
    print(nm,'rc=',rc,'maxerr=',(o-ref).abs().max().item())
def bench(fn, blk):
    o=torch.empty_like(x)
    for _ in range(3): fn(x.data_ptr(),g.data_ptr(),b.data_ptr(),o.data_ptr(),rows,h,1e-6,rows,blk)
    t0=time.perf_counter()
    for _ in range(30): fn(x.data_ptr(),g.data_ptr(),b.data_ptr(),o.data_ptr(),rows,h,1e-6,rows,blk)
    return (time.perf_counter()-t0)/30*1000
for _ in range(3): torch.nn.functional.layer_norm(x,(h,),g,b,1e-6)
t0=time.perf_counter()
for _ in range(30): torch.nn.functional.layer_norm(x,(h,),g,b,1e-6)
ta=(time.perf_counter()-t0)/30*1000
print('ATen=%.4f naive=%.4f shared=%.4f warp=%.4f warp1=%.4f'%(
    ta, bench(naive,256), bench(shared,256), bench(warp,256), bench(warp1,32)))

In [ ]:
# 4/4 - summary
print('If warp maxerr small and warp ms < shared ms -> warp-shuffle wins among TKV kernels.')